In [1]:
import random
import numpy as np
import torch
import time
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

In [2]:
SEED = 42

EPOCHS = 10
BATCH_SIZE = 64
LR = 1e-3

ATOL = 1e-7
RTOL = 1e-5

In [3]:
def set_seed(seed: int | None = None) -> None:
    if seed is None:
        seed = time.time_ns() % (2**32)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [4]:
class SimpleClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int = 20,
        hidden_dim: int = 32,
        num_classes: int = 3,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

In [5]:
def generate_dataset(seed: int = SEED):
    generator = torch.Generator().manual_seed(seed)

    n_samples = 1000
    input_dim = 20
    num_classes = 3

    X = torch.randn(n_samples, input_dim, generator=generator)

    teacher_weights = torch.randn(input_dim, num_classes, generator=generator)

    teacher_bias = torch.randn(num_classes, generator=generator)

    logits = X @ teacher_weights + teacher_bias
    y = logits.argmax(dim=1)

    return X, y

In [6]:
def make_loaders(seed: int = SEED):
    X, y = generate_dataset(seed)

    train_dataset = TensorDataset(X[:800], y[:800])
    val_dataset = TensorDataset(X[800:], y[800:])

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
        num_workers=0,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    return train_loader, val_loader

In [7]:
def compare_state_dicts(
    state_a: dict[str, torch.Tensor], state_b: dict[str, torch.Tensor]
):
    if state_a.keys() != state_b.keys():
        return False

    results = {}

    for key in state_a:
        results[key] = torch.allclose(
            state_a[key],
            state_b[key],
            atol=ATOL,
            rtol=RTOL,
        )

    return results

In [8]:
def print_comparison(title: str, results: bool | dict[str, bool]):
    print(f"\n{title}")

    if isinstance(results, bool):
        print("PASS" if results else "FAIL")
        return

    for name, passed in results.items():
        print(f"{name:40s}: {'PASS' if passed else 'FAIL'}")

In [9]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

In [10]:
from templates.lit import LitModel


def train_model(seed: int = SEED):
    train_loader, val_loader = make_loaders(seed)

    # Симулируем случайное состояние генераторов
    set_seed()

    model = LitModel(
        model=SimpleClassifier,
        seed=seed,
        lr=LR,
        weight_decay=1e-4,
        input_dim=20,
        hidden_dim=32,
        num_classes=3,
    )

    initial_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    metrics = model.fit(
        train_loader=train_loader,
        validation_loader=val_loader,
        epochs=EPOCHS,
        patience=EPOCHS + 1,
    )

    final_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.eval()

    X, _ = generate_dataset(seed)

    with torch.no_grad():
        predictions = model(X.to(model.device)).cpu()

    return {
        "model": model,
        "initial_state": initial_state,
        "final_state": final_state,
        "predictions": predictions,
        "metrics": metrics,
    }

In [11]:
print("Training run #1...")
run_1 = train_model(SEED)

print("\nTraining run #2...")
run_2 = train_model(SEED)

initial_comparison = compare_state_dicts(
    run_1["initial_state"],
    run_2["initial_state"],
)

print_comparison(
    "Initial weights comparison",
    initial_comparison,
)

# Сравнение весов после обучения
final_comparison = compare_state_dicts(
    run_1["final_state"],
    run_2["final_state"],
)

print_comparison(
    "Final weights comparison",
    final_comparison,
)

# Сравнение предсказаний
predictions_equal = torch.allclose(
    run_1["predictions"],
    run_2["predictions"],
    atol=ATOL,
    rtol=RTOL,
)

print_comparison(
    "Predictions comparison",
    predictions_equal,
)

metric_columns = [
    "train_loss",
    "val_loss",
    "train_accuracy",
    "val_accuracy",
]

metrics_1 = run_1["metrics"].groupby("epoch")[metric_columns].mean()

metrics_2 = run_2["metrics"].groupby("epoch")[metric_columns].mean()

metrics_equal = metrics_1.shape == metrics_2.shape and torch.allclose(
    torch.tensor(metrics_1.to_numpy()),
    torch.tensor(metrics_2.to_numpy()),
    atol=ATOL,
    rtol=RTOL,
)

print_comparison("Metrics comparison", metrics_equal)

all_checks_passed = (
    all(initial_comparison.values())
    and all(final_comparison.values())
    and predictions_equal
    and metrics_equal
)

print("\n" + "=" * 50)
print(
    "REPRODUCIBILITY TEST:",
    "PASSED" if all_checks_passed else "FAILED",
)
print("=" * 50)

assert all_checks_passed, "Reproducibility test failed!"

Training run #1...


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

... (17 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Initial weights comparison
model.network.0.weight                  : PASS
model.network.0.bias                    : PASS
model.network.2.weight                  : PASS
model.network.2.bias                    : PASS

Final weights comparison
model.network.0.weight                  : PASS
model.network.0.bias                    : PASS
model.network.2.weight                  : PASS
model.network.2.bias                    : PASS

Predictions comparison
PASS

Metrics comparison
PASS

REPRODUCIBILITY TEST: PASSED
